In [15]:
# ======================= CELL 1: Install & authenticate =======================
# !pip install datasets pandas sacrebleu sentencepiece huggingface_hub

from huggingface_hub import login
from google.colab import userdata

# Authenticate with HuggingFace (reads token from Colab secrets sidebar)
# Make sure you have a secret named HF_TOKEN with your HuggingFace token
login(token=userdata.get('HF_TOKEN'))
print("✓ Authenticated with HuggingFace")

✓ Authenticated with HuggingFace


In [16]:
# ======================= CELL 2: Imports & setup ==============================
import json
import re
import hashlib
import random
import subprocess
from collections import Counter
from pathlib import Path
from datasets import load_dataset
import pandas as pd

random.seed(42)
OUTPUT_DIR = Path("./processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)
print("✓ Setup complete")

✓ Setup complete


In [17]:
# ======================= CELL 3: Download both datasets =======================

# --- Sunbird SALT ---
print("Downloading Sunbird SALT dataset...")
salt = load_dataset("Sunbird/salt", "text-all", split="train")
print(f"  ✓ SALT loaded: {len(salt)} rows")
print(f"  Columns: {salt.column_names}")

# --- Kimera English-Luganda corpus ---
print("\nDownloading Kimera English-Luganda corpus...")
subprocess.run(
    ["git", "clone", "https://github.com/kimrichies/English-Luganda-Parallel-corpus.git", "./kimera_data"],
    capture_output=True
)

kimera_pairs = []
kimera_path = Path("./kimera_data")

# Show what files exist so we know what we're working with
print("  Files found in repo:")
for f in sorted(kimera_path.rglob("*.*")):
    if f.suffix in [".txt", ".csv", ".tsv", ".json", ".xlsx"]:
        print(f"    {f.relative_to(kimera_path)} ({f.stat().st_size / 1024:.1f} KB)")

# Load text files — try multiple encodings since some files are UTF-16
for f in kimera_path.rglob("*.txt"):
    for encoding in ["utf-8", "utf-16", "utf-16-le", "utf-16-be", "latin-1"]:
        try:
            count_before = len(kimera_pairs)
            with open(f, "r", encoding=encoding) as fh:
                for line in fh:
                    line = line.strip()
                    if "\t" in line:
                        parts = line.split("\t", 1)
                        if len(parts) == 2 and parts[0].strip() and parts[1].strip():
                            kimera_pairs.append({"en": parts[0].strip(), "lg": parts[1].strip()})
            added = len(kimera_pairs) - count_before
            if added > 0:
                print(f"  ✓ {f.name}: loaded {added} pairs (encoding: {encoding})")
            break
        except (UnicodeDecodeError, UnicodeError):
            continue

# Load CSV files
for f in kimera_path.rglob("*.csv"):
    for encoding in ["utf-8", "utf-16", "latin-1"]:
        try:
            df = pd.read_csv(f, encoding=encoding)
            en_col = next((c for c in df.columns if c.lower() in ["english", "en", "eng", "source"]), None)
            lg_col = next((c for c in df.columns if c.lower() in ["luganda", "lg", "lug", "target"]), None)
            if en_col and lg_col:
                count_before = len(kimera_pairs)
                for _, row in df.iterrows():
                    if pd.notna(row[en_col]) and pd.notna(row[lg_col]):
                        kimera_pairs.append({"en": str(row[en_col]).strip(), "lg": str(row[lg_col]).strip()})
                added = len(kimera_pairs) - count_before
                print(f"  ✓ {f.name}: loaded {added} pairs (encoding: {encoding})")
            break
        except (UnicodeDecodeError, UnicodeError):
            continue

# Load Excel files if any
for f in kimera_path.rglob("*.xlsx"):
    try:
        df = pd.read_excel(f)
        en_col = next((c for c in df.columns if "eng" in c.lower() or "en" == c.lower() or "source" in c.lower()), None)
        lg_col = next((c for c in df.columns if "lug" in c.lower() or "lg" == c.lower() or "target" in c.lower()), None)
        if en_col and lg_col:
            count_before = len(kimera_pairs)
            for _, row in df.iterrows():
                if pd.notna(row[en_col]) and pd.notna(row[lg_col]):
                    kimera_pairs.append({"en": str(row[en_col]).strip(), "lg": str(row[lg_col]).strip()})
            added = len(kimera_pairs) - count_before
            print(f"  ✓ {f.name}: loaded {added} pairs")
    except Exception as e:
        print(f"  ✗ {f.name}: {e}")

print(f"\n  Total Kimera pairs: {len(kimera_pairs)}")

  ✓ SALT loaded: 23947 rows
  Columns: ['id', 'teo_text', 'swa_text', 'eng_source_text', 'nyn_text', 'ibo_text', 'ach_text', 'lgg_text', 'lug_text', 'ttj_text', 'xog_text', 'eng_target_text']

  Files found in repo:
    Bible_paralle_corpus.xlsx (3747.3 KB)
    FinalMergedDataset5.txt (9136.6 KB)
  ✓ FinalMergedDataset5.txt: loaded 41070 pairs (encoding: utf-16)

  Total Kimera pairs: 41070


In [18]:
# ======================= CELL 4: Extract pairs from SALT ======================

print("Extracting English-Luganda pairs from SALT...")

salt_pairs = []
for row in salt:
    en = row.get("eng_source_text") or row.get("eng_target_text")
    lg = row.get("lug_text")

    if en and lg and isinstance(en, str) and isinstance(lg, str):
        en = en.strip()
        lg = lg.strip()
        if en and lg:
            salt_pairs.append({"en": en, "lg": lg})

print(f"  ✓ Extracted {len(salt_pairs)} English-Luganda pairs from SALT")

# Show a few examples so you can verify quality
print("\n  Sample pairs:")
for i, p in enumerate(salt_pairs[:3]):
    print(f"    EN: {p['en'][:80]}")
    print(f"    LG: {p['lg'][:80]}")
    print()

Extracting English-Luganda pairs from SALT...
  ✓ Extracted 23947 English-Luganda pairs from SALT

  Sample pairs:
    EN: Eggplants always grow best under warm conditions.
    LG: Bbiringanya lubeerera  asinga kukulira mu mbeera ya bugumu

    EN: Farmland is sometimes a challenge to farmers.
    LG: Ettaka ly'okulimirako n'okulundirako ebiseera ebimu kisoomooza abalimi

    EN: Farmers should be encouraged to grow more coffee.
    LG: Abalimi balina okukubirizibwa okwongera okulima emmwanyi



In [19]:
# ======================= CELL 5: Combine all pairs ============================

all_pairs = []
for p in salt_pairs:
    all_pairs.append({**p, "source": "salt"})
for p in kimera_pairs:
    all_pairs.append({**p, "source": "kimera"})

print(f"Total raw pairs: {len(all_pairs)}")
print(f"  From SALT:   {len(salt_pairs)}")
print(f"  From Kimera: {len(kimera_pairs)}")

Total raw pairs: 65017
  From SALT:   23947
  From Kimera: 41070


In [23]:
# ======================= CELL 6: Clean the data ===============================

cleaned_pairs = []
rejected = Counter()
seen_hashes = set()

def clean_text(text):
    """Basic text cleaning."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"([.!?])\1{3,}", r"\1\1\1", text)
    return text

def is_valid_pair(en, lg):
  """Filter out bad pairs."""
  if not en or not lg:
      return False
  if len(en) < 3 or len(lg) < 3:
      return False
  if len(en) > 500 or len(lg) > 500:
      return False
  ratio = len(en) / max(len(lg), 1)
  if ratio > 5 or ratio < 0.2:
      return False
  if re.match(r"^[\d\s\W]+$", en) or re.match(r"^[\d\s\W]+$", lg):
      return False
  if en.lower().strip() == lg.lower().strip():
      return False
  return True

  cleaned_pairs = []
rejected = Counter()
seen_hashes = set()

for pair in all_pairs:
    en = clean_text(pair["en"])
    lg = clean_text(pair["lg"])

    # Deduplicate
    pair_hash = hashlib.md5(f"{en.lower()}|||{lg.lower()}".encode()).hexdigest()
    if pair_hash in seen_hashes:
        rejected["duplicate"] += 1
        continue
    seen_hashes.add(pair_hash)

    if is_valid_pair(en, lg):
        cleaned_pairs.append({"en": en, "lg": lg, "source": pair["source"]})
    else:
        if not en or not lg:
            rejected["empty"] += 1
        elif len(en) < 3 or len(lg) < 3:
            rejected["too_short"] += 1
        elif len(en) > 500 or len(lg) > 500:
            rejected["too_long"] += 1
        else:
            rejected["other"] += 1

print(f"{'='*50}")
print(f"CLEANING RESULTS")
print(f"{'='*50}")
print(f"Input pairs:    {len(all_pairs)}")
print(f"Clean pairs:    {len(cleaned_pairs)}")
print(f"Rejected:       {sum(rejected.values())}")
for reason, count in rejected.most_common():
    print(f"  - {reason}: {count}")

source_counts = Counter(p["source"] for p in cleaned_pairs)
print(f"\nBy source:")
for src, count in source_counts.items():
    print(f"  {src}: {count}")

CLEANING RESULTS
Input pairs:    65017
Clean pairs:    42019
Rejected:       22998
  - duplicate: 22996
  - other: 2

By source:
  salt: 23943
  kimera: 18076


In [24]:
# ======================= CELL 7: Format for fine-tuning =======================

formatted_examples = []

# --- Translation templates (varied so the model doesn't memorize one phrasing) ---
templates_en_to_lg = [
    "Translate the following English text to Luganda:\n{en}",
    "Convert this to Luganda:\n{en}",
    "English to Luganda:\n{en}",
    "How do you say this in Luganda?\n{en}",
    "Translate to Luganda:\n{en}",
]

templates_lg_to_en = [
    "Translate the following Luganda text to English:\n{lg}",
    "Convert this to English:\n{lg}",
    "Luganda to English:\n{lg}",
    "What does this mean in English?\n{lg}",
    "Translate to English:\n{lg}",
]

for pair in cleaned_pairs:
    # English → Luganda
    t = random.choice(templates_en_to_lg)
    formatted_examples.append({
        "instruction": t.format(en=pair["en"]),
        "output": pair["lg"],
        "task": "translation_en_lg",
    })

    # Luganda → English
    t = random.choice(templates_lg_to_en)
    formatted_examples.append({
        "instruction": t.format(lg=pair["lg"]),
        "output": pair["en"],
        "task": "translation_lg_en",
    })

# --- Conversational templates (teach the model to respond IN Luganda) ---
conversation_templates = [
    ("Yogera mu Luganda: {en}", "{lg}"),
    ("Nkusaba oddemu mu Luganda. {en}", "{lg}"),
    ("Respond in Luganda: {en}", "{lg}"),
    ("Answer in Luganda:\n{en}", "{lg}"),
]

conv_subset = random.sample(cleaned_pairs, min(len(cleaned_pairs), len(cleaned_pairs) // 2))
for pair in conv_subset:
    tmpl_in, tmpl_out = random.choice(conversation_templates)
    formatted_examples.append({
        "instruction": tmpl_in.format(en=pair["en"], lg=pair["lg"]),
        "output": tmpl_out.format(lg=pair["lg"], en=pair["en"]),
        "task": "conversation",
    })

random.shuffle(formatted_examples)

print(f"Total formatted examples: {len(formatted_examples)}")
task_counts = Counter(ex["task"] for ex in formatted_examples)
for task, count in task_counts.items():
    print(f"  {task}: {count}")



Total formatted examples: 105047
  translation_en_lg: 42019
  conversation: 21009
  translation_lg_en: 42019


In [25]:
# ======================= CELL 8: Train / Val / Test split =====================

n = len(formatted_examples)
train_end = int(n * 0.90)
val_end = int(n * 0.95)

train_data = formatted_examples[:train_end]
val_data = formatted_examples[train_end:val_end]
test_data = formatted_examples[val_end:]

print(f"Data split:")
print(f"  Train:      {len(train_data)} examples")
print(f"  Validation: {len(val_data)} examples")
print(f"  Test:       {len(test_data)} examples")


# ======================= CELL 9: Save as JSONL ================================

def to_chat_format(example):
    """Convert to chat message format for SFTTrainer."""
    return {
        "messages": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }


def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(to_chat_format(item), ensure_ascii=False) + "\n")
    print(f"  ✓ Saved {len(data)} examples → {filepath}")


print("Saving datasets...")
save_jsonl(train_data, OUTPUT_DIR / "train.jsonl")
save_jsonl(val_data, OUTPUT_DIR / "val.jsonl")
save_jsonl(test_data, OUTPUT_DIR / "test.jsonl")

# Also save clean pairs for inspection
with open(OUTPUT_DIR / "all_clean_pairs.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_pairs, f, ensure_ascii=False, indent=2)
    print(f"  ✓ Saved {len(cleaned_pairs)} clean pairs → all_clean_pairs.json")

Data split:
  Train:      94542 examples
  Validation: 5252 examples
  Test:       5253 examples
Saving datasets...
  ✓ Saved 94542 examples → processed_data/train.jsonl
  ✓ Saved 5252 examples → processed_data/val.jsonl
  ✓ Saved 5253 examples → processed_data/test.jsonl
  ✓ Saved 42019 clean pairs → all_clean_pairs.json


In [26]:
# ======================= CELL 10: Quality check ===============================

print(f"\n{'='*50}")
print("SAMPLE TRAINING EXAMPLES")
print(f"{'='*50}")

for i, ex in enumerate(random.sample(train_data, min(5, len(train_data)))):
    print(f"\n--- Example {i+1} ({ex['task']}) ---")
    print(f"  USER:      {ex['instruction'][:120]}")
    print(f"  ASSISTANT: {ex['output'][:120]}")

# Length stats
en_lengths = [len(p["en"].split()) for p in cleaned_pairs]
lg_lengths = [len(p["lg"].split()) for p in cleaned_pairs]

print(f"\n{'='*50}")
print("LENGTH STATISTICS (words)")
print(f"{'='*50}")
print(f"English — avg: {sum(en_lengths)/len(en_lengths):.1f}, "
      f"min: {min(en_lengths)}, max: {max(en_lengths)}, "
      f"median: {sorted(en_lengths)[len(en_lengths)//2]}")
print(f"Luganda — avg: {sum(lg_lengths)/len(lg_lengths):.1f}, "
      f"min: {min(lg_lengths)}, max: {max(lg_lengths)}, "
      f"median: {sorted(lg_lengths)[len(lg_lengths)//2]}")

print(f"\n{'='*50}")
print("✓ DATA PIPELINE COMPLETE!")
print(f"{'='*50}")
print(f"Files saved in: {OUTPUT_DIR}/")
print(f"  train.jsonl  — {len(train_data)} examples")
print(f"  val.jsonl    — {len(val_data)} examples")
print(f"  test.jsonl   — {len(test_data)} examples")
print(f"\nNext step: upload these to Google Drive so they persist,")
print(f"then we move to Phase 4 (choosing the base model).")


SAMPLE TRAINING EXAMPLES

--- Example 1 (translation_en_lg) ---
  USER:      How do you say this in Luganda?
We need to support our locally manufactured products.
  ASSISTANT: twetaaga okuwagira ebyamaguzi ebikoleddwa kuno.

--- Example 2 (translation_lg_en) ---
  USER:      Translate the following Luganda text to English:
Abategesi bakozesa bubi ssente z'ekkanisa.
  ASSISTANT: The organizers are mismanaging the church funds.

--- Example 3 (translation_en_lg) ---
  USER:      How do you say this in Luganda?
Most corruption cases get thrown out of court due to lack of evidence
  ASSISTANT: Emisango gy'obuli bw'enguzi egisinga gigobebwa mu mbuga z'amateeka olw'okubulwa obukakafu.

--- Example 4 (translation_lg_en) ---
  USER:      Luganda to English:
Ssaabawabuzi wa gavumenti ajja kugonjoola ebizibu ebikwata ku ntobazi.
  ASSISTANT: The solicitor general will solve disputes over wetlands.

--- Example 5 (conversation) ---
  USER:      Answer in Luganda:
A businessman has given some of 

In [27]:
# Mount Google Drive and copy processed data
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = "/content/drive/MyDrive/luganda_llm_data"
shutil.copytree("./processed_data", drive_path, dirs_exist_ok=True)
print(f"✓ Data saved to Google Drive: {drive_path}")

Mounted at /content/drive
✓ Data saved to Google Drive: /content/drive/MyDrive/luganda_llm_data
